# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaivarshini-06/flyrank-ml-internship-jaivarshini/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

This rule uses two signals: staleness and search volume. Content that has not been updated for at least 180 days and has at least 300 impressions in the last 90 days will be prioritized for refresh review.

The reason code will be `stale_visible` when both conditions are met, and `below_rule_threshold` otherwise. The action will be `refresh_review` or `monitor`.

I will inspect how both signals relate to the observed decline indicator before applying the rule. The indicator is used only for the signal audit, not for scoring.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

# Locate the repository and dataset
repo_dir = Path.cwd()

while repo_dir != repo_dir.parent and not (
    repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"
).exists():
    repo_dir = repo_dir.parent

data_path = repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"

if not data_path.exists():
    repo_dir = Path("/content/flyrank-ml-internship-jaivarshini")

    if not data_path.exists():
        subprocess.run([
            "git", "clone",
            "https://github.com/Jaivarshini-06/flyrank-ml-internship-jaivarshini.git",
            str(repo_dir)
        ], check=True)

    data_path = repo_dir / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

required = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

# Used only for signal auditing, NOT scoring
df["audit_declining"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# SIGNAL 1: STALENESS
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, np.inf],
    labels=["0-90 days", "91-180 days", "181+ days"]
)

stale_table = df.groupby(
    "staleness_bucket", observed=False
).agg(
    n=("audit_declining", "size"),
    declining=("audit_declining", "sum"),
    decline_rate=("audit_declining", "mean")
)

stale_table["decline_rate_pct"] = (
    stale_table["decline_rate"] * 100
).round(2)

print("\nSIGNAL 1: STALENESS")
display(stale_table[["n", "declining", "decline_rate_pct"]])

rates = stale_table["decline_rate"].dropna()

if len(rates) < 2:
    stale_verdict = "FALSE"
    stale_explanation = "Not enough populated buckets to compare."
elif rates.iloc[-1] - rates.iloc[0] > 0.02:
    stale_verdict = "CONFIRMED"
    stale_explanation = "The oldest bucket has a higher decline rate."
elif rates.iloc[-1] - rates.iloc[0] < -0.02:
    stale_verdict = "OPPOSITE"
    stale_explanation = "The oldest bucket has a lower decline rate."
else:
    stale_verdict = "MIXED"
    stale_explanation = "The endpoint bucket rates are similar."

print("Verdict:", stale_verdict, "-", stale_explanation)


# SIGNAL 2: SEARCH VOLUME
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 999, np.inf],
    labels=["Below 100", "100-999", "1000+"]
)

volume_table = df.groupby(
    "volume_bucket", observed=False
).agg(
    n=("audit_declining", "size"),
    declining=("audit_declining", "sum"),
    decline_rate=("audit_declining", "mean"),
    median_impressions=("impressions_90d", "median")
)

volume_table["decline_rate_pct"] = (
    volume_table["decline_rate"] * 100
).round(2)

print("\nSIGNAL 2: SEARCH VOLUME")
display(volume_table[
    ["n", "declining", "median_impressions", "decline_rate_pct"]
])

rates = volume_table["decline_rate"].dropna()

if len(rates) < 2:
    volume_verdict = "FALSE"
    volume_explanation = "Not enough populated buckets to compare."
elif rates.iloc[-1] - rates.iloc[0] > 0.02:
    volume_verdict = "CONFIRMED"
    volume_explanation = "The highest-volume bucket has a higher decline rate."
elif rates.iloc[-1] - rates.iloc[0] < -0.02:
    volume_verdict = "OPPOSITE"
    volume_explanation = "The highest-volume bucket has a lower decline rate."
else:
    volume_verdict = "MIXED"
    volume_explanation = "The endpoint bucket rates are similar."

print("Verdict:", volume_verdict, "-", volume_explanation)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.